In [2]:
import pandas as pd

files = {
    "GPT_OSS_RAG": "results/rag_combined_results_gpt_oss_with_rag.csv",
    "GPT_OSS_NO_RAG": "results/rag_combined_results_gpt_oss_without_rag.csv",
    "LLAMA_RAG": "results/rag_combined_results_llama_with_rag.csv",
    "LLAMA_NO_RAG": "results/rag_combined_results_llama_without_rag.csv",
    # "TEST": "results/my_test_llama_with_rag.csv"
}

dfs = {}

for name, file in files.items():
    df = pd.read_csv(file)
    dfs[name] = df

    print("=" * 60)
    print(name)
    print("=" * 60)

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nData types:")
    print(df.dtypes)

    print("\nDescriptive statistics:")
    print(df.describe(include='all'))

GPT_OSS_RAG

Shape:
(572, 13)

Columns:
['context', 'question', 'answer', 'experiment', 'generated_answer', 'retrieved_contexts', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy', 'bertscore_precision', 'bertscore_recall', 'bertscore_f1']

Missing values:
context                 0
question                0
answer                  0
experiment              0
generated_answer        0
retrieved_contexts      0
context_precision       0
context_recall          0
faithfulness            3
answer_relevancy       37
bertscore_precision     0
bertscore_recall        0
bertscore_f1            0
dtype: int64

Data types:
context                 object
question                object
answer                  object
experiment              object
generated_answer        object
retrieved_contexts      object
context_precision      float64
context_recall         float64
faithfulness           float64
answer_relevancy       float64
bertscore_precision    float64
bertscore_reca

In [7]:
df = pd.read_csv("results/rag_combined_results_llama_with_rag_nan_fixed.csv")

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 572 entries, 0 to 571
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   context             572 non-null    object 
 1   question            572 non-null    object 
 2   answer              572 non-null    object 
 3   experiment          572 non-null    object 
 4   generated_answer    572 non-null    object 
 5   retrieved_contexts  572 non-null    object 
 6   context_precision   572 non-null    float64
 7   context_recall      572 non-null    float64
 8   faithfulness        567 non-null    float64
 9   answer_relevancy    571 non-null    float64
dtypes: float64(4), object(6)
memory usage: 44.8+ KB


In [4]:
df_oss = pd.read_csv(files["GPT_OSS_RAG"])
df_llama = pd.read_csv(files["LLAMA_RAG"])

In [12]:
import pandas as pd

csv1_path = "results/rag_combined_results_llama_with_rag_nan_fixed.csv"   # hasil re-run / file yang ingin jadi sumber pengisi
csv2_path = "results/rag_combined_results_llama_with_rag.csv"   # file utama (mis. ada bertscore)
out_path  = "results/results_llama_with_rag.csv"

df1 = pd.read_csv(csv1_path)
df2 = pd.read_csv(csv2_path)

# Kunci baris (sesuaikan jika perlu)
key_cols = ["question", "answer", "experiment"]

# Kolom metrik yang ingin diisi NaN-nya dari csv1
metric_cols = ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]

# Cek duplicate key biar merge aman
if df1.duplicated(key_cols).any() or df2.duplicated(key_cols).any():
    raise ValueError("Ada duplicate key di salah satu CSV. Perbaiki dulu key_cols.")

# Merge metric dari df1 ke df2
tmp = df1[key_cols + metric_cols].copy()
tmp = tmp.rename(columns={c: f"{c}__src" for c in metric_cols})

merged = df2.merge(tmp, on=key_cols, how="left")

# Isi NaN di df2 dari df1
for c in metric_cols:
    merged[c] = merged[c].fillna(merged[f"{c}__src"])
    merged.drop(columns=[f"{c}__src"], inplace=True)

print("Null counts setelah merge:")
print(merged.isnull().sum())

merged.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


Null counts setelah merge:
context                0
question               0
answer                 0
experiment             0
generated_answer       0
retrieved_contexts     0
context_precision      0
context_recall         0
faithfulness           5
answer_relevancy       0
bertscore_precision    0
bertscore_recall       0
bertscore_f1           0
dtype: int64
Saved: results/results_llama_with_rag.csv


In [10]:
df.isna().sum()

context               0
question              0
answer                0
experiment            0
generated_answer      0
retrieved_contexts    0
context_precision     0
context_recall        0
faithfulness          5
answer_relevancy      1
dtype: int64

In [9]:
df_llama.isna().sum()

context                 0
question                0
answer                  0
experiment              0
generated_answer        0
retrieved_contexts      0
context_precision       0
context_recall          6
faithfulness           15
answer_relevancy        3
bertscore_precision     0
bertscore_recall        0
bertscore_f1            0
dtype: int64

In [5]:
df_oss.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 572 entries, 0 to 571
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   context              572 non-null    object 
 1   question             572 non-null    object 
 2   answer               572 non-null    object 
 3   experiment           572 non-null    object 
 4   generated_answer     572 non-null    object 
 5   retrieved_contexts   572 non-null    object 
 6   context_precision    572 non-null    float64
 7   context_recall       572 non-null    float64
 8   faithfulness         569 non-null    float64
 9   answer_relevancy     535 non-null    float64
 10  bertscore_precision  572 non-null    float64
 11  bertscore_recall     572 non-null    float64
 12  bertscore_f1         572 non-null    float64
dtypes: float64(7), object(6)
memory usage: 58.2+ KB


In [30]:
nan_rows = df[df["answer_relevancy"].isna()].copy()

print("NaN answer_relevancy:", len(nan_rows))
print("retrieved_contexts kosong:", (nan_rows["retrieved_contexts"].astype(str).isin(["[]", "", "nan"])).sum())
print("generated_answer kosong:", nan_rows["generated_answer"].fillna("").str.strip().eq("").sum())

# lihat sampel
display(nan_rows[["question"]].head(10))

NaN answer_relevancy: 37
retrieved_contexts kosong: 0
generated_answer kosong: 0


,question
7,Pasien dengan Jenis kelamin: Perempuan Usia: 9...
21,Pasien dengan Jenis kelamin: Perempuan Usia: 1...
22,Pasien dengan Jenis kelamin: Perempuan Usia: 1...
27,Pasien dengan Jenis kelamin: Laki-laki Usia: 3...
28,Pasien dengan Jenis kelamin: Laki-laki Usia: 3...
34,Pasien dengan Jenis kelamin: Perempuan Usia: 3...
42,Pasien dengan Jenis kelamin: Perempuan Usia: 8...
55,Pasien dengan Jenis kelamin: Perempuan Usia: 2...
76,Pasien dengan Jenis kelamin: Laki-laki Usia: 5...
91,Pasien dengan Jenis kelamin: Laki-laki Usia: 1...


In [33]:
nan_rows["question"][:1]

7    Pasien dengan Jenis kelamin: Perempuan Usia: 9...
Name: question, dtype: object